# Neural Fortran Examples: Reading Layer Weights

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/network_parameters.f90`. It builds the same `input(3)`, `dense(5)`, `dense(2)` stack as the simple example and prints the weights. It does not train. The weights are one draw of the library's random initializer, so a later run prints different numbers. The listing on this page is the draw from this machine.

## Walking the layers

```{literalinclude} Fortran_Code/Section_Examples_Network_Parameters/app/network_parameters.f90
:language: fortran
:lines: 10-31
```

`net % layers` is an array of layer objects. `layers(n) % name` is the layer name, and `layers(n) % p` is the polymorphic payload. `select type` recovers the concrete type.

For a `dense_layer`, `weights` has shape $(n_{\mathrm{in}}, n_{\mathrm{out}})$ and `biases` has length $n_{\mathrm{out}}$. The program prints the shape and the weight values. It does not print the biases. The `conv2d_layer` branch is present so the same walk would print a convolution kernel, but this network has no convolution, so that branch does not run. The input layer hits `class default` and is reported as having no parameters.

Fortran prints a rank-2 array in column-major order. For the $3 \times 5$ weight matrix, the first three printed values are the first column, which is one output unit. The figure reshapes that stream with Fortran order, so each column of the picture is one output unit and each row is one input.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Network_Parameters/app/network_parameters.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Network_Parameters/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `network_parameters` from the release build. Standard error was empty and the process exited 0.

```{literalinclude} Fortran_Code/Section_Examples_Network_Parameters/network_parameters.out
:language: text
```

## The printed weights

![Random dense weights](images/nf_network_parameters.png)

Layer 2 is $3 \times 5$ and has five biases, which are not in the print. Layer 3 is $5 \times 2$ and has two biases. The values sit on both sides of zero, which is what a small random normal draw looks like before any call to `backward`.

`get_params`, on the next page, returns weights and biases in one vector. This program is the lower-level view: the fields inside `dense_layer`.


## Conclusion

The useful Fortran feature here is `select type`. The network stores every layer as a `layer` with a class pointer, and the example recovers `dense_layer` when it wants the weight array. The convolution branch shows the same pattern for a kernel.